In [ ]:
import mysql.connector
import time
import logging
import csv
import os
from datetime import datetime, timezone
from getpass import getpass

# ---------------------------------------------------------
# LOGGING
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# ---------------------------------------------------------
# DB CONNECTION
# ---------------------------------------------------------
db = mysql.connector.connect(
    host="localhost",
    user="Oyeniyi_ETL",
    password=getpass("Enter MySQL password: "),
    database="electricity_capstone",
    autocommit=False
)
cur = db.cursor()

batch_id    = int(time.time())
pipeline_id = "etl_v1"
run_ts      = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

logging.info(f"PIPELINE START | batch_id={batch_id} | pipeline_id={pipeline_id}")

# ---------------------------------------------------------
# LOAD DIMENSIONS
# ---------------------------------------------------------
logging.info("Loading dim_time...")
cur.execute("SELECT datetime_utc, time_id FROM dim_time")
time_map = {dt.strftime("%Y-%m-%d %H:%M:%S"): tid for dt, tid in cur.fetchall()}

logging.info("Loading dim_country...")
cur.execute("SELECT country_code, country_id FROM dim_country")
code_map = {(cc or '').strip().upper(): cid for cc, cid in cur.fetchall()}

cur.execute("SELECT country_name, country_id FROM dim_country")
name_map = {(cn or '').strip(): cid for cn, cid in cur.fetchall()}

# ---------------------------------------------------------
# ROW COUNTS
# ---------------------------------------------------------
cur.execute("SELECT COUNT(*) FROM tmp_cleaned_demand")
demand_total = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM tmp_cleaned_price")
price_total = cur.fetchone()[0]

CHUNK = 200_000

# ---------------------------------------------------------
# PROGRESS TRACKER
# ---------------------------------------------------------
def progress(label, processed, total, inserted, skipped, start_ts):
    elapsed = time.time() - start_ts
    pct = processed / total * 100
    rate = processed / elapsed if elapsed > 0 else 0
    eta = (total - processed) / rate if rate > 0 else 0
    logging.info(
        f"{label} | {processed:,}/{total:,} ({pct:5.1f}%) "
        f"inserted={inserted:,} skipped={skipped:,} "
        f"rate={rate:,.0f} rows/s elapsed={elapsed:,.1f}s eta={eta:,.1f}s"
    )

# ---------------------------------------------------------
# FACT DEMAND LOADER
# ---------------------------------------------------------
def load_fact_demand():
    logging.info("Loading fact_demand...")
    offset = processed = inserted = skipped = 0
    start_ts = time.time()

    while True:
        cur.execute(f"""
            SELECT datetime_utc, country_code, cov_ratio, value, value_scaled
            FROM tmp_cleaned_demand
            ORDER BY datetime_utc
            LIMIT {CHUNK} OFFSET {offset}
        """)
        rows = cur.fetchall()
        if not rows:
            break

        batch = []
        for dt_utc, iso2, cov, val, val_scaled in rows:
            processed += 1
            dt_key = dt_utc.strftime("%Y-%m-%d %H:%M:%S")
            iso_key = (iso2 or '').strip().upper()

            time_id = time_map.get(dt_key)
            country_id = code_map.get(iso_key)

            if not time_id or not country_id:
                skipped += 1
                continue

            batch.append((time_id, country_id, cov, val, val_scaled,
                          batch_id, pipeline_id, run_ts))

        if batch:
            cur.executemany("""
                INSERT IGNORE INTO fact_demand (
                    time_id, country_id, cov_ratio, value, value_scaled,
                    batch_id, pipeline_id, load_ts
                ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            """, batch)
            db.commit()
            inserted += cur.rowcount

        if processed % 50_000 == 0:
            progress("fact_demand", processed, demand_total, inserted, skipped, start_ts)

        offset += CHUNK

    progress("fact_demand FINAL", processed, demand_total, inserted, skipped, start_ts)

# ---------------------------------------------------------
# FACT PRICE LOADER
# ---------------------------------------------------------
def load_fact_price():
    logging.info("Loading fact_price...")
    offset = processed = inserted = skipped = 0
    start_ts = time.time()

    while True:
        cur.execute(f"""
            SELECT datetime_utc, country_name, price_eur_mwhe
            FROM tmp_cleaned_price
            ORDER BY datetime_utc
            LIMIT {CHUNK} OFFSET {offset}
        """)
        rows = cur.fetchall()
        if not rows:
            break

        batch = []
        for dt_utc, cname, price in rows:
            processed += 1
            dt_key = dt_utc.strftime("%Y-%m-%d %H:%M:%S")
            cname_key = (cname or '').strip()

            time_id = time_map.get(dt_key)
            country_id = name_map.get(cname_key)

            if not time_id or not country_id:
                skipped += 1
                continue

            batch.append((time_id, country_id, price,
                          batch_id, pipeline_id, run_ts))

        if batch:
            cur.executemany("""
                INSERT IGNORE INTO fact_price (
                    time_id, country_id, price_eur_mwhe,
                    batch_id, pipeline_id, load_ts
                ) VALUES (%s,%s,%s,%s,%s,%s)
            """, batch)
            db.commit()
            inserted += cur.rowcount

        if processed % 50_000 == 0:
            progress("fact_price", processed, price_total, inserted, skipped, start_ts)

        offset += CHUNK

    progress("fact_price FINAL", processed, price_total, inserted, skipped, start_ts)

# ---------------------------------------------------------
# DIAGNOSTIC MODULE (CSV + AUDIT TABLES)
# ---------------------------------------------------------
def run_diagnostics():
    diag_dir = "etl_diagnostics"
    os.makedirs(diag_dir, exist_ok=True)
    run_ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    # 1. Summary
    cur.execute("SELECT COUNT(*) FROM tmp_cleaned_demand")
    staging_rows = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM fact_demand")
    fact_rows = cur.fetchone()[0]

    skipped_rows = staging_rows - fact_rows

    with open(os.path.join(diag_dir, f"summary_{batch_id}.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        writer.writerow(["staging_rows", staging_rows])
        writer.writerow(["fact_rows", fact_rows])
        writer.writerow(["skipped_rows", skipped_rows])

    # 2. Invalid ISO2
    cur.execute("""
        SELECT d.country_code, COUNT(*)
        FROM tmp_cleaned_demand d
        LEFT JOIN dim_country c
            ON UPPER(d.country_code) = UPPER(c.country_code)
        WHERE c.country_id IS NULL
        GROUP BY d.country_code
        ORDER BY COUNT(*) DESC
    """)
    invalid_iso2 = cur.fetchall()

    with open(os.path.join(diag_dir, f"invalid_iso2_{batch_id}.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["invalid_iso2", "row_count"])
        writer.writerows(invalid_iso2)

    # 3. Invalid timestamps
    cur.execute("""
        SELECT d.datetime_utc, COUNT(*)
        FROM tmp_cleaned_demand d
        LEFT JOIN dim_time t
            ON d.datetime_utc = t.datetime_utc
        WHERE t.time_id IS NULL
        GROUP BY d.datetime_utc
        ORDER BY COUNT(*) DESC
    """)
    invalid_ts = cur.fetchall()

    with open(os.path.join(diag_dir, f"invalid_timestamps_{batch_id}.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["invalid_timestamp", "row_count"])
        writer.writerows(invalid_ts)

    # 4. Duplicates
    cur.execute("""
        SELECT t.time_id, c.country_id, COUNT(*) AS occurrences
        FROM tmp_cleaned_demand d
        JOIN dim_time t
            ON d.datetime_utc = t.datetime_utc
        JOIN dim_country c
            ON UPPER(d.country_code) = UPPER(c.country_code)
        GROUP BY t.time_id, c.country_id
        HAVING COUNT(*) > 1
        ORDER BY occurrences DESC
    """)
    duplicates = cur.fetchall()

    with open(os.path.join(diag_dir, f"duplicates_{batch_id}.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["time_id", "country_id", "occurrences"])
        writer.writerows(duplicates)

    # Write audit metadata
    cur.execute("""
        INSERT INTO etl_batch_log (
            batch_id, pipeline_id, start_ts, end_ts, status, row_count, message
        ) VALUES (%s, %s, %s, NOW(), %s, %s, %s)
    """, (
        batch_id, pipeline_id, run_ts, "diagnostics_complete",
        fact_rows, f"Skipped rows: {skipped_rows}"
    ))

    for iso2, count in invalid_iso2:
        cur.execute("""
            INSERT INTO etl_audit (table_name, batch_id, pipeline_id, rows_inserted, run_ts, message)
            VALUES ('fact_demand', %s, %s, %s, NOW(), %s)
        """, (batch_id, pipeline_id, count, f"Invalid ISO2: {iso2}"))

    for ts, count in invalid_ts:
        cur.execute("""
            INSERT INTO etl_audit (table_name, batch_id, pipeline_id, rows_inserted, run_ts, message)
            VALUES ('fact_demand', %s, %s, %s, NOW(), %s)
        """, (batch_id, pipeline_id, count, f"Invalid timestamp: {ts}"))

    for time_id, country_id, occurrences in duplicates:
        cur.execute("""
            INSERT INTO etl_audit (table_name, batch_id, pipeline_id, rows_inserted, run_ts, message)
            VALUES ('fact_demand', %s, %s, %s, NOW(), %s)
        """, (batch_id, pipeline_id, occurrences - 1,
              f"Duplicate key (time_id={time_id}, country_id={country_id})"))

    db.commit()

    logging.info("Diagnostics complete. CSVs written to etl_diagnostics/")

# ---------------------------------------------------------
# RUN ETL + DIAGNOSTICS
# ---------------------------------------------------------
load_fact_demand()
load_fact_price()
run_diagnostics()

logging.info("PIPELINE COMPLETE.")
cur.close()
db.close()
